# V110 - Assessment without straight-liners

Pre-registered plan: the audit found 197 train users with zero-variance assessments (121 all-max, 76 all-zero) plus 209 cross-split duplicate profiles. These rows contribute no assessment signal and 46 share one vector with different labels, pure noise for assessment heads. Drop zero-variance users from assessment training only, keep them everywhere else. Gate denoised against standard under identical folds with a fresh seed. Same locks.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 3435
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v110")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Denoised vs standard assessment heads

Only training rows change. Evaluation covers all users on identical folds. Chat heads are untouched.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
S = np.array([[num(blobs[u].get(q)) for q in questions] for u in list(train["user_id"])])
S = np.where(np.isnan(S), np.nanmean(S, axis=0), S)
flat = S.std(1) == 0
print(f"straight-liners dropped from training: {flat.sum()} / {n_train}")
A = (S - S.mean(0)) / (S.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(mask, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(A, strata, fold_groups)):
        tr2 = tr[~mask[tr]] if mask is not None else tr
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(A[tr2], Y[tr2, j])
            oof[va, j] = m.predict(A[va])
        fids[va] = fi + 1
    return oof, fids

oof_s, fid = ridge_oof(None)
oof_d, _ = ridge_oof(flat)
base = ndcg5(Y, oof_s)
d = ndcg5(Y, oof_d) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_d[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"standard={base.mean():.5f} denoised={ndcg5(Y, oof_d).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT denoising" if (lo > GATE_FLOOR and wf >= 0) else "KEEP all rows; straight-liners are harmless")

straight-liners dropped from training: 197 / 4000


standard=0.41866 denoised=0.41740 delta=-0.00127 CI95=[-0.00273,+0.00013] worst-fold=-0.01468
KEEP all rows; straight-liners are harmless
